# 🧪 W6-D6 概念实验：亲手搭一个 Function Calling Agent 的每一层

> 配套阅读：`ima/第6周-Day6-搭建Function-Calling-Agent.md`（完整 5 工具教学版代码、生产版改造路径在那边）
> 本 notebook 回答四个问题：
>
> 1. 工具的 **JSON Schema** 怎么从 Python 函数自动生成？参数怎么在执行前校验？
> 2. **决策 → 执行 → 整合** 的 Function Calling 主循环，最短多少行能跑通？
> 3. 多轮对话里"那杭州呢？"为什么能被理解？
> 4. LLM 提取的参数是脏的，**验证-修复回路**怎么救场？
>
> LLM 决策用规则 mock（真实系统只替换 `llm_decide` 一个方法），其余全部真实执行。

## 实验 1：工具注册表——从函数签名自动生成 JSON Schema + 执行前校验

Function Calling 的地基是**工具注册表**：每个工具有 `name + description + parameters schema`。
Schema 不该手写维护——用 `inspect` 从类型注解自动生成，然后写一个 20 行的校验器，
在工具执行**之前**拦截非法参数（安全底线）。

In [ ]:
import inspect, json, re
from typing import get_type_hints

PY2JSON = {str: "string", int: "integer", float: "number", bool: "boolean"}
JSON2PY = {"string": str, "integer": int, "number": (int, float), "boolean": bool}

def build_schema(fn):
    """从函数签名 + 类型注解自动生成 OpenAI 风格的 JSON Schema"""
    hints = get_type_hints(fn)
    props, required = {}, []
    for name, p in inspect.signature(fn).parameters.items():
        props[name] = {"type": PY2JSON[hints[name]]}
        if p.default is inspect.Parameter.empty:      # 没有默认值 → 必填
            required.append(name)
    return {"type": "object", "properties": props, "required": required}

def validate_args(schema, args):
    """迷你校验器：返回错误列表（空列表 = 通过）"""
    errors = []
    for k in schema["required"]:
        if k not in args:
            errors.append(f"缺少必填参数: {k}")
    for k, v in args.items():
        spec = schema["properties"].get(k)
        if spec is None:
            errors.append(f"未知参数: {k}"); continue
        expected = JSON2PY[spec["type"]]
        ok = isinstance(v, expected) and not (spec["type"] != "boolean" and isinstance(v, bool))
        if not ok:
            errors.append(f"参数 {k} 期望 {spec['type']}，收到 {type(v).__name__}")
    return errors

# --- 三个工具（真实函数 + 给 LLM 看的 description）---
WEATHER = {"北京": "32°C 晴", "上海": "28°C 多云", "广州": "35°C 雷阵雨", "杭州": "30°C 阴"}

def get_weather(city: str) -> str:
    """查询指定城市的当前天气"""
    return f"{city}：{WEATHER.get(city, '暂无数据')}"

def calculate(expression: str) -> str:
    """执行数学计算"""
    return f"{expression} = {eval(expression)}"   # 教学演示，生产环境禁用裸 eval

def send_notification(phone: str, message: str) -> str:
    """发送短信通知"""
    if not re.fullmatch(r"1\d{10}", phone):
        raise ValueError(f"手机号格式非法: {phone}")
    return f"短信已发送至 {phone}"

TOOL_REGISTRY = {
    "get_weather":     {"fn": get_weather,     "desc": "查询指定城市的当前天气"},
    "calculate":       {"fn": calculate,       "desc": "执行数学计算"},
    "send_notification": {"fn": send_notification, "desc": "发送短信通知给指定手机号"},
}
for t in TOOL_REGISTRY.values():
    t["schema"] = build_schema(t["fn"])

print("自动生成的 Schema：")
print(json.dumps(TOOL_REGISTRY["send_notification"]["schema"], ensure_ascii=False, indent=2))

print("\n🧪 校验器测试（LLM 吐出的脏参数）：")
dirty_cases = [
    {"phone": "13800138000", "message": "ok"},        # 合法
    {"phone": "138001380000", "message": "ok"},       # 12 位 → 但类型对，schema 层拦不住！
    {"phone": 13800138000, "message": "ok"},          # 类型错误 → 拦住
    {"message": "缺手机号"},                           # 缺必填 → 拦住
]
for args in dirty_cases:
    errs = validate_args(TOOL_REGISTRY["send_notification"]["schema"], args)
    print(f"  {str(args)[:50]:<52s} → {'✅ 通过' if not errs else '❌ ' + '; '.join(errs)}")
print("\n注意：12 位手机号类型是 string，schema 校验拦不住 → 需要正则约束（见实验 4）")

## 实验 2：Function Calling 主循环——决策 → 校验 → 执行 → 整合

整个 Agent 的骨架就是这条流水线。`llm_decide` 是**唯一**需要替换成真实 LLM 的方法
（生产版：把工具 Schema 发给 LLM API，解析它返回的 tool_call）。

In [ ]:
import re
from collections import defaultdict

class FunctionCallingAgent:
    """最小可用的 Function Calling Agent（约 45 行核心逻辑）"""

    def __init__(self, registry):
        self.registry = registry
        self.history = []                       # 对话历史 = 状态管理（内存版）
        self.tool_usage = defaultdict(int)

    def llm_decide(self, msg):
        """mock LLM 决策：生产环境替换为 LLM API + tools schema"""
        if any(w in msg for w in ["天气", "温度", "几度"]):
            m = next((c for c in WEATHER if c in msg), None)
            return ("get_weather", {"city": m} if m else None)
        if any(w in msg for w in ["通知", "短信"]):
            ph = re.search(r"1\d{10}", msg)
            return ("send_notification", {"phone": ph.group() if ph else None,
                                          "message": msg})
        if any(w in msg for w in ["计算", "等于", "多少"]) or re.search(r"[\d+\-*/.() ]{3,}", msg):
            expr = re.search(r"[\d+\-*/.()]+[\d+\-*/.() ]*", msg)
            if expr: return ("calculate", {"expression": expr.group().strip()})
        return (None, None)                     # 不需要工具

    def chat(self, msg):
        self.history.append({"role": "user", "content": msg})
        tool, args = self.llm_decide(msg)

        if tool is None:
            reply = f"（直接回答，无需工具）你好！我能查天气、算数、发短信。"
        else:
            errs = validate_args(self.registry[tool]["schema"], args or {})
            if errs:                                        # 参数问题 → 反馈而不是崩溃
                reply = f"⚠️ 参数不完整：{errs[0]}，请补充信息"
            else:
                try:
                    result = self.registry[tool]["fn"](**args)
                except ValueError as e:                     # 业务校验（如手机号正则）
                    reply = f"⚠️ 执行被拒绝：{e}"
                else:
                    self.tool_usage[tool] += 1
                    reply = f"🔧 {tool} → {result}"
                    self.history.append({"role": "tool", "name": tool, "result": result})
        self.history.append({"role": "assistant", "content": reply})
        return reply

agent = FunctionCallingAgent(TOOL_REGISTRY)
for msg in ["北京今天天气怎么样？",
            "帮我算一下 (128+256)*3 等于多少",
            "你好，你是谁？",
            "发短信通知 13800138000 订单已发货",
            "帮我查下天气",          # 参数缺失
            "发短信通知 98765 出货"]:  # 提取不到手机号
    print(f"👤 {msg}\n🤖 {agent.chat(msg)}\n")

## 实验 3：多轮上下文——"那杭州呢？" 为什么能被理解

这句话里没有"天气"二字，单看这句无法决定调什么工具。靠的是**对话历史**里的"槽位"：
上一轮的意图（查天气）被继承，只替换参数（城市→杭州）。

对比实验：无状态 Agent（每次只看当前消息）vs 带上下文继承的 Agent。

In [ ]:
class ContextAgent(FunctionCallingAgent):
    """升级版：记住最近一次成功的工具调用，意图/槽位缺失时从它继承"""

    def __init__(self, registry):
        super().__init__(registry)
        self.last_call = None                  # (tool, args)

    def llm_decide(self, msg):
        tool, args = super().llm_decide(msg)
        if tool and args and all(args.values()):
            self.last_call = (tool, args)      # 本轮自带完整意图 → 记住
            return tool, args
        # 意图或槽位缺失 → 从上一轮继承
        if tool is None and self.last_call:
            tool = self.last_call[0]            # "那杭州呢？" → 继承意图 get_weather
        if tool and self.last_call:
            merged = dict(self.last_call[1])   # 先继承旧槽位
            if args:                           # 本轮显式给出的槽位优先
                merged.update({k: v for k, v in args.items() if v is not None})
            else:                              # 没有任何槽位 → 从消息里抓（如城市名）
                m = next((c for c in WEATHER if c in msg), None)
                if m and tool == "get_weather":
                    merged["city"] = m
            args = merged if all(merged.values()) else None
        if tool and args and all(args.values()):
            self.last_call = (tool, args)
        return tool, args

# --- 对比实验 ---
print("场景：连续三句话，后两句高度省略")
dialog = ["广州今天天气怎么样？", "那杭州呢？", "上海呢？"]

print("\n❌ 无状态 Agent（每轮只看当前消息）：")
stateless = FunctionCallingAgent(TOOL_REGISTRY)
for m in dialog:
    print(f"  👤 {m}")
    print(f"  🤖 {stateless.chat(m)}")

print("\n✅ 带上下文继承的 Agent：")
ctx_agent = ContextAgent(TOOL_REGISTRY)
for m in dialog:
    print(f"  👤 {m}")
    print(f"  🤖 {ctx_agent.chat(m)}")

print("\n结论：多轮理解的本质不是魔法，是 history 里的意图+槽位继承。")
print("（生产环境这些信息全部进入 LLM 的 messages 上下文，由 LLM 自动完成继承。）")

## 实验 4：参数修复回路——LLM 吐脏参数时，把错误喂回去

md 里的误区三："LLM 提取的参数直接用就行"是错的。真实 Function Calling 的标准做法：

1. LLM 第一次提取 → 校验失败（如手机号 12 位）
2. 把**错误信息**作为反馈喂给 LLM 再试
3. 最多 N 轮，仍失败则向用户澄清

这个"校验→反馈→重提取"的回路，是教学版到生产版最重要的一步。

In [ ]:
import re

def mock_llm_extract(msg, feedback=None):
    """模拟 LLM 参数提取：收到错误反馈后会自我修正"""
    ph = re.search(r"\d{11,12}", msg)          # 故意匹配 11~12 位：第一次可能提错
    phone = ph.group() if ph else "138001380000"
    if feedback:                                 # 修复轮：根据反馈截断/修正
        phone = phone[:11] if len(phone) > 11 else phone
    return {"phone": phone, "message": "您的订单已发货"}

def repair_loop(msg, max_repairs=3):
    """校验 → 反馈 → 重提取 回路"""
    feedback = None
    for attempt in range(1, max_repairs + 1):
        args = mock_llm_extract(msg, feedback)
        errs = validate_args(TOOL_REGISTRY["send_notification"]["schema"], args)
        pattern_err = not re.fullmatch(r"1\d{10}", args["phone"])   # schema 之外的正则约束
        print(f"  第 {attempt} 次提取: phone={args['phone']} "
              f"({'✅通过' if not errs and not pattern_err else '❌schema错误' if errs else '❌正则不匹配'})")
        if not errs and not pattern_err:
            return send_notification(**args)
        feedback = errs[0] if errs else f"手机号必须是11位，当前{len(args['phone'])}位"
        print(f"     ↩️ 反馈给 LLM: {feedback}")
    return "⚠️ 多次修复失败，转人工处理"

print("用户输入：『发短信通知 138001380000 出货』（手机号多了一位）\n")
print("结果:", repair_loop("发短信通知 138001380000 出货"))
print("\n要点：错误信息本身就是 prompt 的一部分——这就是为什么参数错误『不重试但反馈』。")

## 实验 5：运行统计——工具调用分布可视化

跑一批混合请求，看 Agent 的行为画像：多少走工具、多少直接回答、各工具占比。
（对应 md 里"运营报表"的类比：数据驱动的工具集优化——没人用的工具该删，描述不清的工具该改。）

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
from collections import Counter

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

batch = (["北京天气怎么样", "上海天气", "广州天气", "杭州天气", "深圳天气", "天津天气"] * 2 +
         ["计算 128*3", "计算 99+1", "计算 500/4"] +
         ["发短信 13800138000 发货了", "通知 13912345678 到货"] +
         ["你好", "你是谁", "谢谢", "再见", "帮我推荐糖水"])

stat_agent = FunctionCallingAgent(TOOL_REGISTRY)
for m in batch:
    stat_agent.chat(m)

usage = dict(stat_agent.tool_usage)
n_direct = len(batch) - sum(usage.values())
labels = list(usage.keys()) + ["直接回答"]
counts = list(usage.values()) + [n_direct]
colors = ["#42A5F5", "#66BB6A", "#FFA726", "#B0BEC5"]

print(f"总消息 {len(batch)} 条 | 工具调用 {sum(usage.values())} 次 | 直接回答 {n_direct} 次")
print("工具分布:", dict(Counter({k: v for k, v in usage.items()})))

fig, ax = plt.subplots(figsize=(7.5, 4))
bars = ax.bar(labels, counts, color=colors[:len(labels)], alpha=0.88)
for b, c in zip(bars, counts):
    ax.text(b.get_x() + b.get_width()/2, c + 0.3, str(c), ha="center", fontweight="bold")
ax.set_ylabel("调用次数")
ax.set_title(f"Agent 行为画像（{len(batch)} 条请求）")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()
print("这张图回答工程问题：哪个工具最忙（要不要加缓存/限流）、哪些工具没被选中（描述要不要改）。")

## 结论

| 实验 | 验证的概念 | 一句话 |
|---|---|---|
| 1 | Schema 自动生成 + 参数校验 | Schema 从签名生成；校验是执行前的安全底线 |
| 2 | Function Calling 主循环 | 决策→校验→执行→整合，唯一 mock 的是 `llm_decide` |
| 3 | 多轮上下文 | "那杭州呢？"= 历史里的意图继承 + 槽位填充 |
| 4 | 参数修复回路 | 校验失败 → 错误反馈给 LLM → 重新提取 |
| 5 | 运行统计 | 工具调用分布是工具集优化的依据 |

**教学版 → 生产版的改造点只有一个：把 `llm_decide` 换成 LLM API。**其余的校验、上下文、回路都是确定性工程，不依赖模型。

→ 深入阅读：同名 md 第 3-4 节（完整 5 工具版代码、生产版伪代码对照表）